In [ ]:
### Load libraries and set params
library(Matrix)
library(parallel)
library(Seurat)
library(gridExtra)
library(ggplot2)
library(future)
library(ggpubr)
library(viridis)
library(openxlsx)
library(viridis)
library(scran)
library(gghighlight)
library(dplyr)
library(org.Dm.eg.db)
library(ggExtra)
library(scDblFinder)
library(patchwork)
library(RhpcBLASctl)
blas_set_num_threads(18)
library(peakRAM)
options(device=pdf)
options(future.globals.maxSize = 214748364800)
library(future)
plan("multicore", workers = 18)

### Set directories
mainDir <- "/data/ebaird/scRNAseq/SCENTINELsep24/"
repDir <- paste0(mainDir, "subset_reanalysis/")
figDir <- paste0(repDir, "figs/")
tabDir <- paste0(repDir, "tables/")
refsDir <- paste0(mainDir, "refs/")


dir.create(repDir, recursive = TRUE, showWarnings = FALSE)
dir.create(figDir, recursive = TRUE, showWarnings = FALSE)
dir.create(tabDir, recursive = TRUE, showWarnings = FALSE)

### Set colours
mycols <- c(1, '#ffffe5','#fff7bc','#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506')
mycols11 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray")
mycols13 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray", "blue", "green")
mycols17 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray", "blue", "green", rainbow(4))

mycols20 <- c("yellow", '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "chartreuse", "blue", "green", rainbow(4), "darkslategray3", "darksalmon", "darkorchid4", "cyan")

corner <- function(x) x[1:5,1:5]
cols <- c(colorRamps::matlab.like2(20)[1:18], "deeppink2", "deeppink3", "deeppink4")

getdensity <- function(x, y, ...) {
      dens <- MASS::kde2d(x, y, ...)
      ix <- findInterval(x, dens$x)
      iy <- findInterval(y, dens$y)
      ii <- cbind(ix, iy)
      return(dens$z[ii])
}

In [ ]:
# Load object
seu <- readRDS(file = paste0(mainDir, "/composition_DEG_signatures/signatures.rds"))

In [ ]:
### Cluster subsetting
Idents(seu) <- seu$seurat_clusters

# Subsets
subsets <- list(
    neuronal = c("6", "7", "8", "13", "14"),
    panagiotis = c("2", "9", "11", "12"),
    others = c("0", "1", "3", "4", "5", "10")
)

subset_name <- "panagiotis" # Change this to the desired subset name

subset <- subset(seu, idents = subsets[[subset_name]])
subset <- SCTransform(subset, verbose = FALSE, return.only.var.genes = FALSE, conserve.memory = TRUE, vst.flavor = "v1")
subset <- RunPCA(subset, npcs = 30, verbose = FALSE)

# rename seurat_clusters to original_clusters
subset$original_clusters <- subset$seurat_clusters

pdf(paste0(figDir, "elbow_", subset_name, "subset.pdf"))
ElbowPlot(subset, ndims=30)
dev.off()
ElbowPlot(subset, ndims=30)

In [ ]:
subset <- RunUMAP(subset, dims = 1:14, verbose = FALSE)
subset <- FindNeighbors(subset, dims = 1:14, verbose = FALSE)
subset <- FindClusters(subset, resolution = 0.3)

DimPlot(subset, label = TRUE)
ggsave(filename = paste0(figDir, "UMAP.", subset_name, "_subset.jpeg"), plot = last_plot(), width = 8, height = 6)

In [ ]:
# saveRDS(subset, file = paste0(repDir, subset_name, "_subset.rds"))
subset <- readRDS(paste0(repDir, "others_subset.rds"))

In [ ]:
# Trace origin of cells in each cluster of the subset
for (i in levels(Idents(subset))) {
  cells <- WhichCells(subset, idents = i)
  cat("Cluster", i, ":", length(cells), "cells\n")
  
  original_clusters <- seu@meta.data[cells, "seurat_clusters"]
  
  print(table(original_clusters))
}

In [ ]:
### Add new clustering as metadata to full seurat object
new_idents <- Idents(subset)

vector <- rep(NA, length(Cells(seu)))
names(vector) <- Cells(seu)

vector[names(new_idents)] <- as.character(new_idents)

cluster_assignment <- paste0(subset_name,"_clusters")
seu[[cluster_assignment]] <- vector

DimPlot(seu, group.by = paste0(subset_name, "_clusters"), label = TRUE, na.value = "grey") +
  ggtitle(paste0(subset_name, " clusters annotated in full seurat object"))

jpeg(filename = paste0(figDir, subset_name, "_recluster_full_umap.jpeg"), width = 2000, height = 2000, res = 150)
print(DimPlot(seu, group.by = paste0(subset_name, "_clusters"), label = TRUE, na.value = "grey") +
  ggtitle(paste0(subset_name, " clusters annotated in full seurat object")))
dev.off()

In [ ]:
# Find markers for subset
all.markers <- FindAllMarkers(subset, only.pos = TRUE, min.pct = 0.25, logfc.threshold = 0.25, verbose = TRUE)

write.csv(all.markers,file=paste0(tabDir,'allMarkers_others_clusters.csv'))
all.markers %>%
        group_by(cluster) %>%
        slice_max(n = 10, order_by = avg_log2FC) -> top10
write.csv(top10,file=paste0(tabDir,'top10Markers_others_clusters.csv'))

In [ ]:
jpeg(paste0(figDir, 'top10markers.dotplot_others_clusters.jpeg'), quality = 100, width = 2000, height = 1000, res = 150)
print(
  DotPlot(subset, features = unique(top10$gene), dot.scale = 6) + 
  RotatedAxis() +
  theme(
    axis.text.x = element_text(size = 8)
  ) +
  scale_color_gradientn(
    colours = c("white", "forestgreen"),
    limits = c(0, 1.5),
    oob = scales::squish
  ) +
  ggtitle("Top 10 Markers per cluster")
)
dev.off()

In [ ]:
### Dotplot of top markers whole seurat object, only show subset

DefaultAssay(subset)<-'SCT'
Idents(subset) <- paste0('original_clusters')
top10 <- read.csv(paste0(mainDir,'QC_clustering/tables/top10Markers_merged_clusters.csv'))
top10 <- top10[top10$cluster %in% subsets[[subset_name]], ]

jpeg(paste0(figDir,'top10markers.heatmap_', subset_name, '_subset_original_clusters_vsfull.jpeg'), quality = 100)
print(DoHeatmap(subset, features = top10$gene) + NoLegend())
dev.off()

jpeg(paste0(figDir, 'top10markers.dotplot_', subset_name, '_subset_original_clusters_vsfull.jpeg'), quality = 100, width = 2000, height = 1000, res = 150)
print(
  DotPlot(subset, features = unique(top10$gene), dot.scale = 6) + 
  RotatedAxis() +
  theme(
    axis.text.x = element_text(size = 8)
  ) +
  scale_color_gradientn(
    colours = c("white", "forestgreen"),
  ) +
  ggtitle(paste0("Top 10 Markers per cluster ", subset_name, " subset"))
)
dev.off()

In [ ]:
# Save seu with subset clusters
saveRDS(seu, file = paste0(repDir, "subset_reclusters.rds"))